# 02 — Linking every player to his Transfermarkt card

Goal: give each FPL player (season, element) his Transfermarkt player_id. First the clubs and seasons, then the player names.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.load import load_gw, load_valuations, load_players

gw = load_gw()
val = load_valuations()
players = load_players()


## Step 1 — Premier League clubs per season

In [2]:
pl_clubs = gw[["season", "team"]].drop_duplicates()
assert len(pl_clubs) == 100  # 5 seasons x 20 clubs
pl_clubs.head()

,season,team
0,2021-22,Man Utd
1,2021-22,Aston Villa
2,2021-22,Watford
3,2021-22,Southampton
5,2021-22,Liverpool


## Step 2 — Club name mapping (Transfermarkt → FPL)

In [ ]:
club_map = {
    "Arsenal FC": "Arsenal",
    "Aston Villa": "Aston Villa",
    "AFC Bournemouth": "Bournemouth",
    "Brentford FC": "Brentford",
    "Brighton & Hove Albion": "Brighton",
    "Burnley FC": "Burnley",
    "Chelsea FC": "Chelsea",
    "Crystal Palace": "Crystal Palace",
    "Everton FC": "Everton",
    "Fulham FC": "Fulham",
    "Ipswich Town": "Ipswich",
    "Leeds United": "Leeds",
    "Leicester City": "Leicester",
    "Liverpool FC": "Liverpool",
    "Luton Town": "Luton",
    "Manchester City": "Man City",
    "Manchester United": "Man Utd",
    "Newcastle United": "Newcastle",
    "Norwich City": "Norwich",
    "Nottingham Forest": "Nott'm Forest",
    "Sheffield United": "Sheffield Utd",
    "Southampton FC": "Southampton",
    "Sunderland AFC": "Sunderland",
    "Tottenham Hotspur": "Spurs",
    "Watford FC": "Watford",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
}

assert set(club_map.values()) == set(gw["team"].unique())  # every FPL team is mapped

assert set(club_map.keys()) - set(val["current_club_name"].unique()) == set()  # every name in the dict exists in val 

## Step 3 — Season of each valuation

I chose July 1st as the boundary: the season usually ends in May, so June valuations are basically a recap of the season that just finished, and the player is still listed at his old club.

Unfortunately a deal done in late June or a value posted early July can land on the wrong side, but those cases should show up as unmatched and I'll check them by hand.

In [4]:
year = val["date"].dt.year
month = val["date"].dt.month

# season starts this year if month >= 7, otherwise it started last year
start_year = np.where(month >= 7, year, year - 1)

# build "2022-23" from start_year = 2022
start = pd.Series(start_year, index=val.index).astype(str)
end = (pd.Series(start_year, index=val.index) + 1).astype(str).str[2:4]
val["season"] = start + "-" + end

In [5]:
# check the July 1st boundary on a known transfer (Rice: West Ham -> Arsenal, July 2023)
rice_id = players[players["name"] == "Declan Rice"]["player_id"].iloc[0]
val[val["player_id"] == rice_id][["date", "current_club_name", "season"]]

,date,current_club_name,season
29,2021-07-15,West Ham United,2021-22
939,2021-12-23,West Ham United,2021-22
2010,2022-06-15,West Ham United,2021-22
3409,2022-11-03,West Ham United,2022-23
5115,2023-06-20,West Ham United,2022-23
5509,2023-10-09,Arsenal FC,2023-24
6076,2023-12-19,Arsenal FC,2023-24
7246,2024-05-27,Arsenal FC,2023-24
7912,2024-12-16,Arsenal FC,2024-25
9031,2025-05-30,Arsenal FC,2024-25


## Step 4 — Keep only Premier League valuations

The GB1 filter keeps 351 club names (see 01), so I keep a valuation only if its club was in the Premier League that season.

In [ ]:
# Transfermarkt name -> FPL name, NaN if not one of the 27 PL clubs
val["team"] = val["current_club_name"].map(club_map)

# keep a valuation only if its (season, team) pair is in the Premier League that season
val_pl = val.merge(pl_clubs, on=["season", "team"], how="inner")

print(len(val), len(val_pl))
val_pl["season"].value_counts().sort_index()

11267 -> 6992


season
2021-22    1216
2022-23    1481
2023-24    1384
2024-25    1383
2025-26    1528
Name: count, dtype: int64

Testing on a club that got relegated (Leeds, in the Championship in 2023-24 and 2024-25): in val_pl we only get the seasons where Leeds was in the PL.

In [7]:
print(val[val["team"] == "Leeds"]["season"].value_counts().sort_index())
print(val_pl[val_pl["team"] == "Leeds"]["season"].value_counts().sort_index())

season
2021-22    57
2022-23    79
2023-24    60
2024-25    53
2025-26    81
Name: count, dtype: int64
season
2021-22    57
2022-23    79
2025-26    81
Name: count, dtype: int64
